# AI Dance Generation from Music and Text

이 노트북은 음악 파일과 자연어 댄스 스타일 설명을 입력받아 2D 인체 스켈레톤 좌표 시퀀스를 생성하는 AI 모델을 학습하고 실행합니다.

## 주요 기능
- 음악 특징 추출 (BPM, Beat, Chroma, MFCC 등)
- 텍스트 스타일 임베딩 (BERT)
- Transformer 기반 댄스 생성
- 평가 메트릭 (Beat Align Score, FVD, Diversity)

## 참고 논문
- X-Dancer (ICCV 2025)
- DGSDP (2024)
- AIST++ (2021)

## 1. 환경 설정

In [5]:
# GPU 확인
!nvidia-smi

Mon Nov 17 19:59:30 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 프로젝트 클론 (GitHub URL을 본인 저장소로 변경하세요)
!git clone https://github.com/yc9954/yg.git
%cd yg

# 또는 Google Drive 사용
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/yg

In [7]:
# 패키지 설치
!pip install -q torch torchvision torchaudio
!pip install -q librosa soundfile
!pip install -q transformers tokenizers sentencepiece
!pip install -q opencv-python mediapipe pillow
!pip install -q einops tqdm matplotlib seaborn
!pip install -q imageio imageio-ffmpeg scipy

## 2. 더미 데이터셋 생성 (테스트용)

In [ ]:
# 더미 데이터셋 생성
from dance_datasets import create_dummy_dataset

create_dummy_dataset(
    output_dir='/content/dummy_dance_dataset',
    num_samples=50
)

print("✅ 더미 데이터셋 생성 완료!")

## 3. 모델 학습

In [ ]:
# 학습 실행
!python train.py \
  --data_dir /content/dummy_dance_dataset \
  --batch_size 8 \
  --epochs 50 \
  --lr 1e-4 \
  --d_model 256 \
  --num_heads 4 \
  --num_layers 4 \
  --seq_length 64 \
  --output_dir /content/checkpoints \
  --save_interval 10

## 4. 댄스 생성 (추론)

In [ ]:
# 샘플 음악 파일 준비 (업로드 또는 샘플 사용)
# from google.colab import files
# uploaded = files.upload()

# 또는 더미 데이터셋의 오디오 사용
audio_path = '/content/dummy_dance_dataset/audio/audio_0000.wav'
style_text = '강렬한 K-pop 댄스'

# 댄스 생성
!python inference.py \
  --checkpoint /content/checkpoints/best_model.pth \
  --audio_path {audio_path} \
  --style_text "{style_text}" \
  --output_path /content/output/generated_dance.json \
  --fps 30

## 5. 시각화

In [ ]:
# 비디오 생성
!python visualize.py \
  --pose_sequence /content/output/generated_dance.json \
  --output_video /content/output/dance_video.mp4 \
  --audio_path {audio_path} \
  --fps 30 \
  --title "Generated K-pop Dance"

In [ ]:
# Colab에서 비디오 재생
from IPython.display import HTML
from base64 import b64encode

video_path = '/content/output/dance_video.mp4'

with open(video_path, 'rb') as f:
    video_bytes = f.read()

video_b64 = b64encode(video_bytes).decode()

HTML(f'''
<video width="640" height="480" controls>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
''')

## 6. 평가

In [ ]:
# 평가 메트릭 계산
import numpy as np
import librosa
from evaluate import evaluate_generated_dances
from utils import load_pose_sequence

# 생성된 포즈 로드
generated_poses, _ = load_pose_sequence('/content/output/generated_dance.json')
generated_poses = np.expand_dims(generated_poses, 0)  # (1, T, J, D)

# 실제 포즈 (더미 데이터에서 로드)
real_pose_path = '/content/dummy_dance_dataset/poses/pose_0000.npy'
real_poses = np.load(real_pose_path)
real_poses = np.expand_dims(real_poses, 0)  # (1, T, J, D)

# 음악 박자 추출
y, sr = librosa.load(audio_path)
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
beat_times = librosa.frames_to_time(beat_frames, sr=sr)

# 평가
metrics = evaluate_generated_dances(
    real_poses,
    generated_poses,
    beat_times,
    fps=30.0
)

print("\n=== Evaluation Results ===")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

## 7. 결과 다운로드

In [ ]:
# 생성된 파일 다운로드
from google.colab import files

files.download('/content/output/generated_dance.json')
files.download('/content/output/dance_video.mp4')

## 8. 고급: 실제 데이터셋으로 학습

In [ ]:
# AIST++ 데이터셋 다운로드 (선택적)
# 참고: https://google.github.io/aistplusplus_dataset/

# 1. 데이터셋 다운로드 (약 30GB, 시간 소요)
# !wget https://aistdancedb.ongaaccel.jp/v1.0.0/aist_plusplus_final.zip
# !unzip aist_plusplus_final.zip -d /content/aist_plusplus

# 2. 데이터 전처리
# 실제 구현에서는 SMPL 모델로부터 관절 위치 추출 필요

# 3. 학습
# !python train.py \
#   --data_dir /content/aist_plusplus_processed \
#   --batch_size 16 \
#   --epochs 200 \
#   --lr 1e-4 \
#   --output_dir /content/checkpoints_aist

## 9. 유틸리티 함수

In [ ]:
# 오디오 특징 시각화
import matplotlib.pyplot as plt
from utils import AudioFeatureExtractor

extractor = AudioFeatureExtractor()
features = extractor.extract_all_features(audio_path)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# Chroma
axes[0].imshow(features['chroma'], aspect='auto', cmap='viridis')
axes[0].set_title('Chroma Features')
axes[0].set_ylabel('Pitch Class')

# MFCC
axes[1].imshow(features['mfcc'], aspect='auto', cmap='viridis')
axes[1].set_title('MFCC')
axes[1].set_ylabel('Coefficient')

# Onset Strength
axes[2].plot(features['onset_env'])
axes[2].set_title('Onset Strength')
axes[2].set_xlabel('Time (frames)')
axes[2].set_ylabel('Strength')

plt.tight_layout()
plt.show()

print(f"Tempo: {features['tempo']:.2f} BPM")
print(f"Beats: {len(features['beat_frames'])}")

In [ ]:
# 포즈 통계
from utils import compute_bone_lengths, COCO_SKELETON

poses, _ = load_pose_sequence('/content/output/generated_dance.json')
bone_lengths = compute_bone_lengths(poses, COCO_SKELETON)

print(f"Pose shape: {poses.shape}")
print(f"Bone lengths shape: {bone_lengths.shape}")
print(f"Average bone length: {bone_lengths.mean():.4f}")
print(f"Bone length std: {bone_lengths.std():.4f}")

## 10. 모델 체크포인트 저장 (Google Drive)

In [ ]:
# 체크포인트를 Google Drive에 복사
!cp -r /content/checkpoints /content/drive/MyDrive/
!cp -r /content/output /content/drive/MyDrive/

print("Checkpoints and outputs saved to Google Drive!")